# client-sse: The old HTTP transport (legacy)

**Read this last. Nothing new uses SSE: The spec replaced it with Streamable HTTP in version 2025-03-26.** It's here because older servers still use it, and so you can explain the difference.

**SSE (Server-Sent Events) is a standard web feature: The server keeps one HTTP reply open and pushes messages down it, one by one.** MCP's old HTTP transport used SSE for the server's messages, and a second endpoint for the client's:

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
 client                                 server at 127.0.0.1:8050
 GET  /sse, held open        ---------&gt; /sse
      &lt;--- every answer from the server comes down this stream
 POST /messages/, one per message ----&gt; /messages/
      &lt;--- 202 Accepted (the answer comes down the stream instead)
</pre>

## 1. Starting the server

Open a terminal in this folder and start the server with SSE:

<div style="font-size: 0.85em">

```bash
conda activate MCP_env
python server.py sse
```

</div>

## 2. Connecting, listing the tools and calling `add`

`sse_client(url)` is the SSE version of `streamable_http_client`. It gives back only `read` and `write`, with no session ID function. Everything from `ClientSession` down is the same.

In [1]:
from mcp import ClientSession
from mcp.client.sse import sse_client

url = "http://127.0.0.1:8050/sse"

In [2]:
async with sse_client(url) as (read, write):
    async with ClientSession(read, write) as session:
        init_result = await session.initialize()
        tools_result = await session.list_tools()
        result = await session.call_tool("add", {"a": 2, "b": 3})

print(result.content[0].text)

5


## 3. What the server saw

After its startup lines, the server's terminal printed this:

<div style="font-size: 0.85em">

```text
INFO:     127.0.0.1:49332 - "GET /sse HTTP/1.1" 200 OK
INFO:     127.0.0.1:49333 - "POST /messages/?session_id=736c27ccc7a84dc9830957a9dd0b06cc HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:49333 - "POST /messages/?session_id=736c27ccc7a84dc9830957a9dd0b06cc HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:49333 - "POST /messages/?session_id=736c27ccc7a84dc9830957a9dd0b06cc HTTP/1.1" 202 Accepted
Processing request of type ListToolsRequest
INFO:     127.0.0.1:49333 - "POST /messages/?session_id=736c27ccc7a84dc9830957a9dd0b06cc HTTP/1.1" 202 Accepted
Processing request of type CallToolRequest
```

</div>

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
 GET  /sse        200  the stream for the server's answers, held open
 POST /messages/  202  initialize
 POST /messages/  202  notifications/initialized
 POST /messages/  202  tools/list
 POST /messages/  202  tools/call
</pre>

Every POST gets only "202 Accepted". The real answers, like the tool list and the `5`, travel back down the open `GET /sse` stream. The `?session_id=...` part tells the server which open stream belongs to this client.

## 4. Why Streamable HTTP replaced it

- **Two endpoints instead of one:** `/sse` and `/messages/`, where Streamable HTTP has only `/mcp`.
- **A connection that must stay open:** The `GET /sse` stream has to stay open for the whole session. If it drops, the answers have nowhere to go. That also makes SSE hard to run behind load balancers (which spread requests across several copies of a server), because they expect short request and reply.
- **Answers split from questions:** Each POST gets an empty 202, and its real answer arrives on a different connection. Over Streamable HTTP, the answer comes back in the reply to the POST itself.

`client-sse.py` does the same as a script. With `python server.py sse` running, `python client-sse.py` prints:

<div style="font-size: 0.85em">

```text
add
Add two numbers together
5
```

</div>

## Say it out loud

> "SSE was MCP's first HTTP transport. The client held a `GET /sse` stream open for the server's messages, and sent its own as POSTs to `/messages/`, each answered with just 202 Accepted. Streamable HTTP replaced it in 2025-03-26 with one endpoint, `/mcp`, where each POST gets its own answer, so there's no long-lived connection to keep alive. In the client, only the connect line changes."